# Exercise 14: Conversation between AI Agents

This notebook uses the OpenAI API. Install or update the Python package with `%pip install -U openai` if needed, and set `OPENAI_API_KEY` in your environment. Do not paste your API key into this notebook.

In [ ]:
import os
from getpass import getpass
from openai import OpenAI

os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")
client = OpenAI()
MODEL = "gpt-6-luna"
TURNS = 8

In [ ]:
# System prompts
PROF_SYS = (
  "You are Prof. Browder, former Belle II spokesperson and expert in B physics, "
  "detectors, and accelerators. Speak to an undergraduate. "
  "Rules: (1) Use plain, friendly language. (2) Max 3 sentences. "
  "(3) Offer one next practical step or tiny example. (4) End with one short "
  "follow-up question. (5) If student has a clear plan, reply exactly 'DONE'."
)

STU_SYS = (
  "You are an undergraduate student interested in experimental particle physics. "
  "Rules: (1) Be concise (max 3 sentences). (2) Ask concrete questions about what to learn "
  "or how to start. (3) When you feel ready with a clear plan for next steps, reply exactly 'DONE'."
)

INIT_QUESTION = "Hi Professor, I'm new to B physics and Belle II. What should I learn first to get involved?"


In [ ]:
def step(system_prompt: str, history: list[dict]) -> str:
    """Send one agent's own conversation history and return its reply."""
    resp = client.responses.create(
        model=MODEL,
        instructions=system_prompt,
        input=history,
        reasoning={"effort": "none"},
        max_output_tokens=200,
    )
    reply = resp.output_text.strip()
    if not reply:
        raise RuntimeError("The model returned an empty reply.")
    return reply

In [ ]:
def run_dialogue():
    # Initial question from student
    professor_history: list[dict] = [
        {"role": "user",
         "content": (INIT_QUESTION)}
    ]
    student_history: list[dict] = []
    print(f"\n[Student] {INIT_QUESTION}")

    for turn in range(TURNS):
        # Prof. Browder's turn
        prof = step(PROF_SYS, professor_history)
        print(f"\n[Prof. Browder] {prof}")
        professor_history.append({"role": "assistant", "content": prof})
        student_history.append({"role": "user", "content": prof})
        if prof == "DONE":
            print("\nConversation finished.")
            break

        # student's turn
        student = step(STU_SYS, student_history)
        print(f"\n[Student] {student}")
        student_history.append({"role": "assistant", "content": student})
        professor_history.append({"role": "user", "content": student})
        if student == "DONE":
            print("\nConversation finished.")
            break
    else:
        print("\nReached max turns.")

In [ ]:
if __name__ == "__main__":
    run_dialogue()